In [44]:
from dotenv import load_dotenv

# TypedDict allows us to define the structure of the graph state
from typing_extensions import TypedDict

# Optional means the value can be None
# Literal restricts return values to specific strings
from typing import Optional, Literal

from langgraph.graph import StateGraph, START, END
from openai import OpenAI

In [45]:
load_dotenv()
client = OpenAI()

In [46]:
class State(TypedDict):
    user_query: str
    llm_output: Optional[str]
    is_good: Optional[bool]

In [47]:
def chatBot_gpt(state : State):
    print("ChatBot Node ", state)

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        max_tokens=50,
        messages=[
            {"role":"user", "content": state["user_query"]}
        ]
    )

    state["llm_output"] = response.choices[0].message.content

    return state

In [48]:
def evaluate_responses(state : State) -> Literal["ChatBot_Claude", "EndNode"]:
    print("Evaluate Node", state)

    if "4" in state["llm_output"]:
        return "EndNode"
    
    return "ChatBot_Claude"

In [49]:
def chatBot_claude(state : State):
    print("ChatBot Node ", state)

    response = client.chat.completions.create(
        model="anthropic/claude-3-haiku",
        max_tokens=50,
        messages=[
            {"role":"user", "content": state["user_query"]}
        ]
    )

    state["llm_output"] = response.choices[0].message.content

    return state

In [50]:
def endNode(state : State):
    print("End Node", state)
    return state

In [51]:
graph_builder = StateGraph(State)

graph_builder.add_node("ChatBot_GPT", chatBot_gpt)
graph_builder.add_node("ChatBot_Claude", chatBot_claude)
graph_builder.add_node("EndNode", endNode)


In [52]:
graph_builder.add_edge(START, "ChatBot_GPT")
graph_builder.add_conditional_edges("ChatBot_GPT", evaluate_responses)

graph_builder.add_edge("ChatBot_Claude", "EndNode")

graph_builder.add_edge("EndNode", END)

In [53]:
graph = graph_builder.compile()

In [54]:
update_state = graph.invoke(
    {
        'user_query': "Hey, what is 2 + 6 ?"
    }
)


print("\nFinal State:\n", update_state)

ChatBot Node  {'user_query': 'Hey, what is 2 + 6 ?'}
Evaluate Node {'user_query': 'Hey, what is 2 + 6 ?', 'llm_output': '2 + 6 = 8'}
ChatBot Node  {'user_query': 'Hey, what is 2 + 6 ?', 'llm_output': '2 + 6 = 8'}
End Node {'user_query': 'Hey, what is 2 + 6 ?', 'llm_output': '2 + 6 = 8.'}

Final State:
 {'user_query': 'Hey, what is 2 + 6 ?', 'llm_output': '2 + 6 = 8.'}
